# Bibliotecas

In [1]:
#########################
##      MARIADB        ##
#########################
import mariadb


#########################
#    TOMLLIB            #
#########################
import tomllib


#########################
##       PATHLIB       ##
#########################
import pathlib
from pathlib import Path


#########################
##      DATETIME       ##
#########################
import datetime
from datetime import date


#########################
##      pandas         ##
#########################
import pandas as pd


#########################
##      TRACEBACK      ##
#########################
import traceback

# def raiz_do_projeto

In [2]:
def localizar_raiz_projeto():

    pasta_atual = Path.cwd().resolve()

    for pasta in [
        pasta_atual,
        *pasta_atual.parents
    ]:

        if (
            (pasta / "requirements.txt").exists()
            and
            (pasta / "src").exists()
        ):
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )

PROJETO_RH = localizar_raiz_projeto()

CAMINHO_CONFIG = (
    PROJETO_RH /
    "config.toml"
)

print(
    f"Raiz do projeto: {PROJETO_RH}"
)

print(
    f"Config encontrado: {CAMINHO_CONFIG.exists()}"
)

Raiz do projeto: C:\Users\06962589948\Documents\testes-projeto-rh\projeto_rh_beta
Config encontrado: True


# def conexao_db

In [3]:
def conexao_db():
    conn = None
    cursor = None

    try:

        with open(
            CAMINHO_CONFIG,
            "rb"
        ) as arquivo:

            config = tomllib.load(arquivo)

            db = config["risoluto"]

        conn = mariadb.connect(
            host=db["host"],
            port=db["port"],
            database=db["database"],
            user=db["user"],
            password=db["password"]
        )

        cursor = conn.cursor()

        return conn, cursor

    except Exception as erro:

        print(f"Erro ao conectar com o banco de dados: {erro}")

        return None, None

# def tratamento_de_datas

In [4]:
def tratamento_de_datas(valor):

    if valor is None:
        return None

    if pd.isna(valor):
        return None

    if isinstance(valor, datetime.datetime):
        return valor.date()

    if isinstance(valor, date):
        return valor

    valor = str(valor).strip()

    if valor == "":
        return None

    formatos = [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%Y-%m-%d",
        "%Y/%m/%d"
    ]

    for formato in formatos:
        try:
            return datetime.datetime.strptime(valor, formato).date()

        except ValueError:
            continue

    return None

# def tratar_inteiro

In [5]:
def tratar_inteiro(valor):
    # Verifica se o valor é nulo.
    # Exemplos: None, NaN ou pd.NA.
    # Caso seja nulo, retorna None.
    if pd.isna(valor):
        return None

    # Converte o valor para string e remove
    # espaços no início e no final.
    valor = str(valor).strip()

    # Se o valor estiver vazio após remover
    # os espaços, retorna None.
    if valor == "":
        return None

    try:
        # Primeiro converte o valor para float
        # e depois para inteiro.
        #
        # Isso permite tratar valores como:
        # "10"   -> 10
        # "10.0" -> 10
        # 10.0   -> 10
        return int(float(valor))

    # Caso o valor não possa ser convertido
    # para número, retorna None em vez de
    # interromper o programa com um erro.
    except (ValueError, TypeError):
        return None

# def tratar_valor_id

In [6]:
def tratar_valor_id(valor):
    # Verifica se o valor é nulo, como NaN, None ou pd.NA.
    # Se for nulo, retorna None.
    if pd.isna(valor):
        return None

    # Converte o valor para string e remove espaços
    # no início e no final do texto.
    valor = str(valor).strip()

    # Verifica se, após remover os espaços,
    # o valor ficou vazio.
    # Exemplo: "   " vira "".
    if valor == "":
        return None

    # Se o valor for válido, retorna ele como string.
    return valor

# def atualizar_upsert

In [7]:
def atualizar_upsert(
        tabela,
        coluna,
        valor,
        conn,
        cursor
):

    if valor is None:
        return None

    query = f"""
        INSERT INTO `{tabela}` (`{coluna}`)
        VALUES (?)
        ON DUPLICATE KEY UPDATE
            id = LAST_INSERT_ID(id)
    """

    cursor.execute(query, (valor,))

    return cursor.lastrowid

# def inserir_dados_pk_usuario

    - Quem chama a função: executar_query()

    - O que entra na função: Os dados extraidos pelo web scraping

    - O que a função faz: Cria uma query SQL responsavel por inserir ou atualizar os dados
    do servidor na tabela pk_usuario

    - O que sai da função: Retorna o comando SQL armazenado em sql_inserir_dados_pk_usuario


In [8]:
def inserir_dados_pk_usuario():

    sql_inserir_dados_pk_usuario = """
        INSERT INTO pk_usuario (
            matricula,
            nome,
            cpf,
            especialidade
        )
        VALUES (?, ?, ?, ?)

        ON DUPLICATE KEY UPDATE
            nome = VALUES(nome),
            cpf = VALUES(cpf),
            especialidade = VALUES(especialidade)
    """

    return sql_inserir_dados_pk_usuario

# def inserir_sql_por_profissional

    - Quem chama a função: A função executar_query()

    - O que entra na função: Os dados extraidos pelo web scraping

    - O que a função faz: Cria a query SQL responsável por inserir ou atualizar
     os dados detalhados do servidor na tabela tb_fp_por_profissional

    - O que sai da função: Retorna a query SQL armazenado em sql_inserir_sql_por_profissional




In [9]:
def inserir_sql_por_profissional():

    sql_inserir_sql_por_profissional = """
    INSERT INTO tb_fp_por_profissional(
        matricula_data,
        competencia,
        projecao_mensal,
        horas_trabalhadas_competencia,
        dias_faltas,
        horas_faltas,
        horas_adicionais,
        saldo_competencia,
        horas_aprovadas,
        trab_sobre_aviso,
        sobre_aviso,
        horas_plantao,
        unidade,
        matricula,
        especialidade,
        data,
        entrada_1,
        saida_1,
        entrada_2,
        saida_2,
        entrada_3,
        saida_3,
        entrada_4,
        saida_4,
        nao_aprovadas,
        horas_trabalhadas_dia,
        motivo_falta,
        saldo_dia,
        ultima_atualizacao
    )
    VALUES (
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?, ?, ?, ?, ?
    )
    ON DUPLICATE KEY UPDATE
        competencia = VALUES(competencia),
        projecao_mensal = VALUES(projecao_mensal),
        horas_trabalhadas_competencia = VALUES(horas_trabalhadas_competencia),
        dias_faltas = VALUES(dias_faltas),
        horas_faltas = VALUES(horas_faltas),
        horas_adicionais = VALUES(horas_adicionais),
        saldo_competencia = VALUES(saldo_competencia),
        horas_aprovadas = VALUES(horas_aprovadas),
        trab_sobre_aviso = VALUES(trab_sobre_aviso),
        sobre_aviso = VALUES(sobre_aviso),
        horas_plantao = VALUES(horas_plantao),
        unidade = VALUES(unidade),
        matricula = VALUES(matricula),
        especialidade = VALUES(especialidade),
        data = VALUES(data),
        entrada_1 = VALUES(entrada_1),
        saida_1 = VALUES(saida_1),
        entrada_2 = VALUES(entrada_2),
        saida_2 = VALUES(saida_2),
        entrada_3 = VALUES(entrada_3),
        saida_3 = VALUES(saida_3),
        entrada_4 = VALUES(entrada_4),
        saida_4 = VALUES(saida_4),
        nao_aprovadas = VALUES(nao_aprovadas),
        horas_trabalhadas_dia = VALUES(horas_trabalhadas_dia),
        motivo_falta = VALUES(motivo_falta),
        saldo_dia = VALUES(saldo_dia),
        ultima_atualizacao = VALUES(ultima_atualizacao)
    """

    return sql_inserir_sql_por_profissional

# def obter_ids_servidor_geral

    - Quem chama a função: "Montar_dados_linha"

    - O que entra na função: Linha, conn, cursor
    linha: dados gerais do servidor
    conn: conexão com o banco
    cursor: usado para executar comando no banco

    - O que a função faz: Busca ou cria os registros das colunas existentes na tabela do banco
        através da função atualizar_upsert() e obtem o ID correspondente de cada um deles.

    - O que sai da função: Retorna um dicionario com os IDs de cada coluna da tabela


    manter obter ou seria inserir tambem?

In [10]:
def obter_ids_servidor_geral(linha, conn, cursor):

    id_competencia = atualizar_upsert(
        tabela = "pk_competencia",
        coluna = "competencia",
        valor = linha["competencia"],
        conn = conn,
        cursor = cursor
    )

    id_especialidade = atualizar_upsert(
        tabela = "pk_especialidade",
        coluna = "especialidade",
        valor = linha["especialidade"],
        conn = conn,
        cursor = cursor
    )

    id_motivo_falta = atualizar_upsert(
        tabela = "pk_motivo_falta",
        coluna = "motivo_falta",
        valor = linha["motivo_falta"],
        conn = conn,
        cursor = cursor
    )

    id_unidade = atualizar_upsert(
        tabela = "pk_unidade",
        coluna = "unidade",
        valor = linha["unidade"],
        conn = conn,
        cursor = cursor
    )

    return {
        "id_competencia": id_competencia,
        "id_especialidade": id_especialidade,
        "id_motivo_falta": id_motivo_falta,
        "id_unidade": id_unidade,
    }

# def montar_dados_gerais_servidor

    - Quem chama a função: montar_dados_linha()

    - O que entra na função:linha, ids, dados_gerais_servidor
        linha: Contém os dados gerais do servidor que estãos sendo processados
        ids: Recebe o dicionario que foi criado pela função obter_ids_servidor_geral()
        dados_gerais_servidor: lista onde será armazenado a tupla preparada para inserção no banco de dados

    - O que a função faz: Ela organiza e trata os dados do servidor na mesma ordem das colunas da tabela tb_fp_por_profissional.
        Depois adiciona esses dados como uma nova tupla dentro da lista dados_gerais_servidor

    - O que sai da função: A lsita dados_gerais_servidor recebida pela função é atualizada com o novo registro.


    É possivel excluir e fazer esse tratamento em outro local?

In [11]:
def montar_dados_gerais_servidor(
        linha,
        ids_por_profissional,
        dados_gerais_servidor
):

    dados_gerais_servidor.append((

        linha["matricula_data"],

        tratar_inteiro(
            ids_por_profissional["id_competencia"]
        ),

        linha["projecao_mensal"],

        linha["horas_trabalhadas_competencia"],

        linha["dias_faltas"],

        linha["horas_faltas"],

        linha["horas_adicionais"],

        linha["saldo_competencia"],

        linha["horas_aprovadas"],

        linha["trab_sobre_aviso"],

        linha["sobre_aviso"],

        linha["horas_plantao"],

        tratar_inteiro(
            ids_por_profissional["id_unidade"]
        ),

        tratar_inteiro(
            linha["matricula"]
        ),

        tratar_inteiro(
            ids_por_profissional["id_especialidade"]
        ),

        tratamento_de_datas(
            linha["data"]
        ),

        linha["entrada_1"],

        linha["saida_1"],

        linha["entrada_2"],

        linha["saida_2"],

        linha["entrada_3"],

        linha["saida_3"],

        linha["entrada_4"],

        linha["saida_4"],

        linha["nao_aprovadas"],

        linha["horas_trabalhadas_dia"],

        tratar_inteiro(
            ids_por_profissional["id_motivo_falta"]
        ),

        linha["saldo_dia"],

        linha["ultima_atualizacao"]

    ))

# def montar_dados_servidor

    - Quem chama a função: A função montar_dados_linha()

    - O que entra na função:linha, ids, dados_servidor
        linha: contém os dados do servidor que estão sendo processados
        ids: Dicionario ja criado pela função obter_ids_servidor_geral()
            utilizando aqui para buscar o ID de especialidade e tratar ele.
        
    - O que a função faz: Pega da linha matricula, nome, cpf do servidor,
        pega do dicionarios ids o ID de especialidade, junta os quatro valores em uma tupla
        e adiciona essa tupla na lista dados_servidor

    - O que sai da função: A lista dados_servidor recebida pela função é atualizada com o novo registro.


    resumindo, primeiramente transformamos cada coluna da tabela em uma linha pura, 
    em seguida criamos essa função para tratar os dados de cada linha e atualizar eles dentro dessa lista dados_servidor.

In [12]:
def montar_dados_servidor(
        linha,
        id_especialidade,
        dados_servidor
):

    dados_servidor.append((

        tratar_inteiro(linha["matricula"]),

        linha["nome"],

        linha["cpf"],

        tratar_inteiro(id_especialidade)

    ))

# def montar_dados_linha

    - Quem chama a função: Será chamada pela função principal que percorre os servidores

    - O que entra: linhas, conn, cursor e as listas de dados
    linhas: dados do servidor
    conn: conexao ao db
    cursor: executa os comandos SQL
    lista de dados: dados_do_servidor_geral e dados_do_servidor

    - O que a função faz: Atua como um coordenador de dados.
        Busca os IDs necessarios e chama montar_dados_servidor() e montar_dados_gerais_servidor()
        para preparar os registros.
        Ela não insere diretamente no banco, apenas organiza e prepara os dados para que em seguida 
        a função executar_query() faça a inserção no banco.

    - O que sai: Não retorna valor, apenas atualiza as listas com os dados preparados.

In [13]:
"""def montar_dados_linha(
        linha_servidor_geral,
        linha_servidor,
        conn,
        cursor,
        dados_do_servidor_geral,
        dados_do_servidor
):

    for i in 10:
        print("################")

    # Busca ou cria os IDs relacionados aos dados por profissional:
    # competência, unidade, especialidade e motivo da falta.
    ids_por_profissional = obter_ids_servidor_geral(
        linha=linha_servidor_geral,
        conn=conn,
        cursor=cursor
    )

    # Monta os dados que serão inseridos
    # na tabela tb_fp_por_profissional.
    montar_dados_gerais_servidor(
        linha=linha_servidor_geral,
        ids_por_profissional=ids_por_profissional,
        dados_gerais_servidor=dados_do_servidor_geral
    )

    # Monta os dados que serão inseridos
    # na tabela pk_usuario.
    # Para essa tabela, precisamos apenas do ID da especialidade.
    montar_dados_servidor(
        linha=linha_servidor,
        id_especialidade=ids_por_profissional["id_especialidade"],
        dados_servidor=dados_do_servidor
    )"""

'def montar_dados_linha(\n        linha_servidor_geral,\n        linha_servidor,\n        conn,\n        cursor,\n        dados_do_servidor_geral,\n        dados_do_servidor\n):\n\n    for i in 10:\n        print("################")\n\n    # Busca ou cria os IDs relacionados aos dados por profissional:\n    # competência, unidade, especialidade e motivo da falta.\n    ids_por_profissional = obter_ids_servidor_geral(\n        linha=linha_servidor_geral,\n        conn=conn,\n        cursor=cursor\n    )\n\n    # Monta os dados que serão inseridos\n    # na tabela tb_fp_por_profissional.\n    montar_dados_gerais_servidor(\n        linha=linha_servidor_geral,\n        ids_por_profissional=ids_por_profissional,\n        dados_gerais_servidor=dados_do_servidor_geral\n    )\n\n    # Monta os dados que serão inseridos\n    # na tabela pk_usuario.\n    # Para essa tabela, precisamos apenas do ID da especialidade.\n    montar_dados_servidor(\n        linha=linha_servidor,\n        id_especialida

# def verificar_listas_vazias

    - Quem chama a função: executar_query()

    - O que entra: as daus listas com os dados preparados para inserção

    -O que faz: verifica se as duas listas estão vazias

    - O que sai: Retorna True se ambas estiverem vazias ou retorna False se existir algum dado em uma das listas

In [14]:
def verificar_listas_vazias(
        dados_do_servidor,
        dados_do_servidor_geral
):

    # Retorna True se ambas as listas estiverem vazias
    # Caso pelo menos uma das listas possua dados, retorna False.
    return (
        not dados_do_servidor
        and
        not dados_do_servidor_geral
    )

    # se der True ira infomrar na função executar_query que não foi encontrado nenhum servidor

# def executar_query

    - Quem chama: arquivo web scraping, É executado após ocorrer o web screping

    - O que entra: a conexao com o banco e as listas com os dados preparados pelo scraping

    - O que faz: Verifica se existem dados para inserir, 
        executa os comandos SQL de pk_usuario e tb_fp_por_profissional,
        confirma as alteações com commit() e 
        em caso de erro, desfaz as alterações com rollback()

    - O que sai: Retorna True se os dados forem inseridos, retorna False se não houver dados ou ocorrer algum erro



In [15]:
def executar_query(
        cursor,
        conn,
        dados_do_servidor_geral,
        dados_do_servidor
):

    try:

        # Verifica se não existem dados preparados para inserção.
        if verificar_listas_vazias(
            dados_do_servidor,
            dados_do_servidor_geral
        ):

            print("Nenhum servidor encontrado para inserir.")

            return False

        # Obtém o SQL responsável pela tabela pk_usuario.
        sql_inserir_dados_pk_usuario = inserir_dados_pk_usuario()

        # Obtém o SQL responsável pela tabela tb_fp_por_profissional.
        sql_inserir_por_profissional = inserir_sql_por_profissional()

        # Insere os dados dos servidores na tabela pk_usuario.
        if dados_do_servidor:

            cursor.executemany(
                sql_inserir_dados_pk_usuario,
                dados_do_servidor
            )

        # Insere os dados detalhados na tabela tb_fp_por_profissional.
        if dados_do_servidor_geral:

            cursor.executemany(
                sql_inserir_por_profissional,
                dados_do_servidor_geral
            )

        # Confirma todas as alterações realizadas no banco.
        conn.commit()

        print(
            f"{len(dados_do_servidor)} registro(s) "
            "de servidor processado(s) no banco com sucesso."
        )

        print(
            f"{len(dados_do_servidor_geral)} registro(s) "
            "de dados gerais processado(s) no banco com sucesso."
        )

        return True

    except Exception as erro:

        # Desfaz as alterações caso ocorra algum erro.
        conn.rollback()

        print(
            f"Erro ao executar query: "
            f"{type(erro).__name__}: {erro}"
        )

        return False

# def fechar_conexao

     Fehcha a conexão após executar o codigo todo

In [16]:
def fechar_conexxao(conn, cursor):

    if cursor:
        cursor.close()

    if conn:
        conn.close()

# Criação de busca por servidor

## def buscar_servidor

    - Quem chama: A função é chamada na interface em streamlit (arquivo app.py)

    - o que entra: O nome ou a matricula informada pelo usuario.

    - O que faz: Abre a conexao com o banco e procura pelo servidor em pk_usuario e em tb_fp_por_profissional
        para obtera data mais rescente registrada para aquela matricula.

    - O que sai: Retorna os servidores encontrados, se ocorrer algum ero, retorna uma lista vazia.

In [17]:
def buscar_servidor(valor_busca):

    # Inicializa as variáveis de conexão como None.
    # Isso permite que o bloco finally tente fechar
    # a conexão com segurança mesmo se ocorrer algum erro antes.
    conn = None
    cursor = None

    try:
        # Abre a conexão com o banco de dados
        # e obtém o cursor para executar os comandos SQL.
        conn, cursor = conexao_db()

        # Converte o valor pesquisado para string
        # e remove espaços no início e no final.
        valor_busca = str(valor_busca).strip()

        # Consulta SQL utilizada para localizar o servidor.
        sql = """
        SELECT
            u.matricula,
            u.nome,
            u.cpf,

            -- Busca a data mais recente existente
            -- para aquela matrícula.
            MAX(fp.data) AS ultima_data

        FROM pk_usuarios AS u

        -- Faz a ligação entre a tabela de usuários
        -- e a tabela de frequência pela matrícula.
        LEFT JOIN tb_fp_por_profissional AS fp
            ON fp.matricula = u.matricula

        WHERE
            -- Permite pesquisar diretamente pela matrícula.
            u.matricula = ?

            OR

            -- Permite pesquisar pelo nome ou por parte dele.
            -- LOWER deixa a comparação sem diferença
            -- entre letras maiúsculas e minúsculas.
            LOWER(u.nome) LIKE LOWER(?)

        -- Como está sendo utilizada a função MAX(),
        -- os demais campos precisam ser agrupados.
        GROUP BY
            u.matricula,
            u.nome,
            u.cpf

        -- Ordena os resultados primeiro pelo nome
        -- e depois pela matrícula.
        ORDER BY
            u.nome,
            u.matricula
        """

        # Executa a consulta SQL passando os valores
        # separadamente, evitando montar o SQL manualmente.
        cursor.execute(
            sql,
            (
                # Primeiro ?:
                # comparação exata com a matrícula.
                valor_busca,

                # Segundo ?:
                # busca o texto em qualquer posição do nome.
                f"%{valor_busca}%"
            )
        )

        # Recupera todos os registros encontrados
        # pela consulta.
        resultados = cursor.fetchall()

        # Retorna os resultados encontrados.
        return resultados

    except Exception as erro:
        # Caso ocorra qualquer erro durante a conexão,
        # consulta ou leitura dos resultados,
        # exibe o tipo e a descrição do erro.
        print(
            f"Erro ao buscar servidor: "
            f"{type(erro).__name__}: {erro}"
        )

        # Retorna uma lista vazia para indicar
        # que nenhum resultado pôde ser obtido.
        return []

    finally:
        # Este bloco sempre será executado,
        # independentemente de a consulta funcionar
        # ou ocorrer algum erro.
        #
        # Fecha o cursor e a conexão com o banco.
        fechar_conexxao(
            conn=conn,
            cursor=cursor
        )


## testes

In [18]:
resultado = buscar_servidor(
    valor_busca = "maria nazare"
)

print(resultado)

[(39578, 'MARIA NAZARE GALLOTTI MACIEL GOULART', '42963117949', datetime.date(2026, 6, 30))]


In [19]:
resultado = buscar_servidor(
    valor_busca = "39578"
)

print(resultado)


[(39578, 'MARIA NAZARE GALLOTTI MACIEL GOULART', '42963117949', datetime.date(2026, 6, 30))]
